# Wordle solver benchmark

The solver plays every possible answer (the 3,624-word madalynrose list), guessing from the 9,795-word joined list. This notebook plots the results.

The numbers come from `benchmark.py`, which takes about 10 minutes and writes `results/strategies.csv` and `results/openers.csv`. This notebook only reads those files and draws the charts, so it runs in seconds.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from wordle_solver import WordleSolver, ALPHABET

strategies = pd.read_csv('results/strategies.csv')
openers = pd.read_csv('results/openers.csv', keep_default_na=False)

## 1. Strategies

Each strategy picks every guess, including the first. Games continue past six guesses, so the full distribution is visible; **won** counts games solved within six.

In [ ]:
table = strategies[['strategy', 'win_rate', 'mean_guesses', 'worst']].copy()
table['win_rate'] = (100 * table['win_rate']).round(2).astype(str) + '%'
table['mean_guesses'] = table['mean_guesses'].round(3)
table.rename(columns={'win_rate': 'won in 6', 'mean_guesses': 'average guesses', 'worst': 'worst game'})

In [ ]:
def parse(dist):
    return {int(k): int(v) for k, v in (pair.split(':') for pair in dist.split())}

dists = pd.DataFrame({row.strategy: parse(row.distribution) for row in strategies.itertuples()}).fillna(0).sort_index()
ax = dists.plot(kind='bar', figsize=(10, 4), width=0.85)
ax.axvline(5.5, color='gray', linestyle='--', linewidth=1)  # between 6 and 7 guesses
ax.text(5.6, ax.get_ylim()[1] * 0.9, 'lost', color='gray')
ax.set_xlabel('guesses to find the word')
ax.set_ylabel('games')
ax.set_title('Guesses needed, by strategy (3,624 games each)')
plt.tight_layout()
plt.show()

## 2. The heat map: where each letter sits

This is the statistic behind the `heatmap` strategy and the MadalynRose v1 notebook: the share of possible answers with each letter in each slot.
- `s` starts 17% of answers and ends 33%, because the list includes plurals.
- `e` is the fourth letter of 23%.
- `y` ends 9%.

In [ ]:
solver = WordleSolver()
codes = solver._codes
heat = pd.DataFrame(
    [np.bincount(codes[:, slot], minlength=26) / len(codes) for slot in range(5)],
    index=[f'slot {i}' for i in range(1, 6)], columns=list(ALPHABET),
)
plt.figure(figsize=(13, 3.2))
sns.heatmap(heat, cmap='viridis', cbar_kws={'label': 'share of answers'})
plt.title('Letter frequency by position, 3,624 possible answers')
plt.tight_layout()
plt.show()

## 3. Best starting words

**One-step scores.** Every allowed guess, scored as an opener against all 3,624 answers:
- *entropy*: the expected bits of information;
- *expected left*: how many candidates are expected to remain;
- *heatmap*: the heat-map score.

In [ ]:
cols = ['entropy_rank', 'word', 'entropy', 'expected_left', 'heatmap', 'candidate', 'frequency']
openers[cols].head(15).round(3)

**Full games.** The top entropy openers, popular human openers, and the vowel-heavy words hand-checked in the NLTK CSV's `In Word List` column. Each one is played as the first guess in all 3,624 games, and the entropy strategy picks every guess after that.

In [ ]:
played = openers[openers['mean_guesses'].astype(str) != ''].copy()
for c in ['mean_guesses', 'win_rate', 'worst']:
    played[c] = pd.to_numeric(played[c])
played = played.sort_values('mean_guesses')
show = played[['word', 'entropy_rank', 'entropy', 'mean_guesses', 'win_rate', 'worst']].copy()
show['win_rate'] = (100 * show['win_rate']).round(2).astype(str) + '%'
show.round(3).reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(played['entropy'], played['mean_guesses'])
for row in played.itertuples():
    ax.annotate(row.word, (row.entropy, row.mean_guesses), xytext=(4, 2), textcoords='offset points', fontsize=9)
ax.set_xlabel('opener entropy (bits)')
ax.set_ylabel('average guesses over all answers')
ax.set_title('More information up front means fewer guesses')
plt.tight_layout()
plt.show()

## 4. Vowel-heavy openers

Words like `adieu` and `audio` are popular openers because they test four vowels at once. Vowels are common, so they rarely come back gray and split the answers into few groups. The entropy chart above shows the cost: they score well below consonant-heavy openers like `tares`.

In [ ]:
vowel = openers[openers['word'].isin(['adieu', 'audio', 'ourie', 'auloi', 'yowie', 'youse', 'noyau'])]
vowel[['word', 'entropy_rank', 'entropy', 'expected_left']].round(3).sort_values('entropy_rank')